# The Shakespearean Scholar - Inference Notebook

**Assignment 2: Containerized RAG System**

**Group Members:** [Fill in your names and roll numbers]

This notebook demonstrates the outputs for each phase of the assignment.

## Setup and Imports

In [ ]:
import sys
import json
import jsonlines
from pathlib import Path
import requests
import pandas as pd

# Add paths
sys.path.append('src')
sys.path.append('api')
sys.path.append('evaluation')

print("✅ Setup complete")

## Phase 1: Data ETL & Chunking

### Task 1.1: PDF Extraction and Cleaning

In [ ]:
from A2_etl_chunking import JuliusCaesarETL

# Initialize ETL
etl = JuliusCaesarETL(
    pdf_path="data/julius-caesar.pdf",
    output_path="data/processed_chunks.jsonl"
)

# Run ETL pipeline
chunks = etl.run_pipeline()

print(f"\n✅ Created {len(chunks)} chunks")

### Task 1.2: Display Sample Chunks

In [ ]:
# Display sample chunks
etl.print_sample_chunks(n=3)

# Show chunk statistics
chunk_types = {}
for chunk in chunks:
    chunk_types[chunk.type] = chunk_types.get(chunk.type, 0) + 1

print("\n📊 Chunk Statistics:")
for chunk_type, count in chunk_types.items():
    print(f"  {chunk_type}: {count}")

### Task 1.3: Chunking Strategy Visualization

In [ ]:
# Analyze chunking strategy
import matplotlib.pyplot as plt

# Chunks by act
act_counts = {}
for chunk in chunks:
    act_counts[chunk.act] = act_counts.get(chunk.act, 0) + 1

plt.figure(figsize=(10, 5))
plt.bar(act_counts.keys(), act_counts.values())
plt.xlabel('Act')
plt.ylabel('Number of Chunks')
plt.title('Chunk Distribution by Act')
plt.show()

print("✅ Chunking strategy visualization complete")

## Phase 2: Indexing & Vector Store

### Task 2.1: Embedding and Indexing

In [ ]:
from A2_indexing import JuliusCaesarIndexer

# Initialize indexer
indexer = JuliusCaesarIndexer(
    chunks_path="data/processed_chunks.jsonl",
    db_path="data/chroma_db",
    embedding_model_name="BAAI/bge-base-en-v1.5"
)

# Run indexing
indexer.run_indexing()

print("\n✅ Indexing complete")

### Task 2.2: Test Retrieval

In [ ]:
# Test retrieval with sample queries
test_queries = [
    "What does the Soothsayer say to Caesar?",
    "Brutus's internal conflict",
    "Caesar's assassination"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    indexer.test_retrieval(query, n_results=3)

### Task 2.3: Collection Statistics

In [ ]:
# Display collection statistics
indexer.print_stats()

## Phase 3 & 4: RAG Pipeline with Prompt Engineering

### Task 3.1: Query the RAG System via API

In [ ]:
# Make sure the API is running (docker-compose up or uvicorn)
API_URL = "http://localhost:8000"

def query_rag(question, n_results=5):
    """Query the RAG API"""
    response = requests.post(
        f"{API_URL}/query",
        json={"query": question, "n_results": n_results}
    )
    return response.json()

# Test query
question = "What does the Soothsayer say to Caesar?"
result = query_rag(question)

print(f"Question: {question}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"\nSources Used: {len(result['sources'])}")
for i, source in enumerate(result['sources'][:3]):
    print(f"\nSource {i+1}:")
    print(f"  Act {source['metadata']['act']}, Scene {source['metadata']['scene']}")
    print(f"  Speaker: {source['metadata']['speaker']}")
    print(f"  Text: {source['chunk'][:100]}...")

### Task 3.2: Multiple Query Examples

In [ ]:
# Test with multiple questions
questions = [
    "Why does Brutus join the conspiracy?",
    "What is Antony's strategy in his funeral speech?",
    "How does Caesar respond to warnings?"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print('='*60)
    
    result = query_rag(question)
    print(f"\nA: {result['answer'][:300]}...\n")

## Phase 5: Containerization

### Task 5.1: Verify Docker Setup

In [ ]:
# Check API health
health_response = requests.get(f"{API_URL}/health")
health_data = health_response.json()

print("API Health Check:")
print(f"  Status: {health_data['status']}")
print(f"  Message: {health_data['message']}")
print(f"  Database Count: {health_data.get('db_count', 0)}")

# Check stats
stats_response = requests.get(f"{API_URL}/stats")
stats = stats_response.json()

print("\nDatabase Statistics:")
print(f"  Total Chunks: {stats['total_chunks']}")
print(f"  Embedding Model: {stats['embedding_model']}")
print(f"  Collection: {stats['collection_name']}")

## Phase 6: Evaluation

### Task 6.1: Load Evaluation Questions

In [ ]:
# Load evaluation questions
with open('evaluation/evaluation.json', 'r') as f:
    eval_questions = json.load(f)

print(f"Total Evaluation Questions: {len(eval_questions)}")
print(f"  Factual (Baseline): 25")
print(f"  Analytical (Custom): {len(eval_questions) - 25}")

# Show sample questions
print("\nSample Factual Question:")
print(f"  Q: {eval_questions[1]['question']}")
print(f"  Expected: {eval_questions[1]['ideal_answer']}")

print("\nSample Analytical Question:")
print(f"  Q: {eval_questions[25]['question']}")
print(f"  Expected: {eval_questions[25]['ideal_answer'][:100]}...")

### Task 6.2: Run Sample Evaluations

In [ ]:
# Evaluate a few questions
sample_indices = [1, 13, 25, 30]  # Mix of factual and analytical

for idx in sample_indices:
    q = eval_questions[idx]
    print(f"\n{'='*60}")
    print(f"Question {idx + 1}: {q['question']}")
    print('='*60)
    
    result = query_rag(q['question'])
    
    print(f"\n✓ Generated Answer:")
    print(result['answer'])
    
    print(f"\n✓ Expected Answer:")
    print(q['ideal_answer'])
    
    print(f"\n✓ Sources: {len(result['sources'])}")

### Task 6.3: Evaluation Metrics Summary

In [ ]:
# Load evaluation results if available
import glob

results_files = glob.glob('evaluation/results/results_*.json')
if results_files:
    latest_results = sorted(results_files)[-1]
    
    with open(latest_results, 'r') as f:
        eval_results = json.load(f)
    
    print(f"Evaluation Results from: {latest_results}")
    print(f"\nTotal Questions Evaluated: {len(eval_results)}")
    
    successful = sum(1 for r in eval_results if r['generated_answer'] != "ERROR: Failed to get response")
    print(f"Successful Responses: {successful}/{len(eval_results)} ({successful/len(eval_results)*100:.1f}%)")
    
    avg_sources = sum(r['num_sources'] for r in eval_results) / len(eval_results)
    print(f"Average Sources per Query: {avg_sources:.1f}")
else:
    print("No evaluation results found. Run: python evaluation/A2_evaluation.py")

## Phase 7: Frontend Demo

### Task 7.1: Frontend Access Information

In [ ]:
print("Streamlit Frontend Access:")
print("  URL: http://localhost:8501")
print("  Start with: streamlit run frontend/A2_frontend.py")
print("  Or via Docker: docker-compose up frontend")

print("\nAPI Documentation:")
print("  Swagger UI: http://localhost:8000/docs")
print("  ReDoc: http://localhost:8000/redoc")

## Summary

### System Architecture Summary

In [ ]:
print("""\n
🎭 THE SHAKESPEAREAN SCHOLAR - SYSTEM SUMMARY
================================================

✅ Phase 1: ETL & Chunking
   - PDF extracted and cleaned
   - Scene-based logical chunking implemented
   - Metadata enrichment complete

✅ Phase 2: Indexing
   - Embeddings: BAAI/bge-base-en-v1.5
   - Vector Store: ChromaDB
   - All chunks indexed with metadata

✅ Phase 3: API Backend
   - FastAPI server operational
   - POST /query endpoint functional
   - Proper error handling implemented

✅ Phase 4: Prompt Engineering
   - Shakespearean Scholar persona defined
   - Gemini 2.0 Flash integration complete
   - Citation formatting implemented

✅ Phase 5: Containerization
   - Dockerfile created
   - docker-compose.yml configured
   - All services containerized

✅ Phase 6: Evaluation
   - 35 test questions (25 factual + 10 analytical)
   - RAGAs metrics framework integrated
   - Evaluation report generated

✅ Phase 7: Frontend
   - Streamlit UI implemented
   - Interactive query interface
   - Source visualization

================================================
""")

## Notes for Demo

**To run the complete system:**

1. Ensure `.env` file has your `GOOGLE_API_KEY`
2. Place `julius-caesar.pdf` in `data/` directory
3. Run: `docker-compose up --build`
4. Access:
   - API: http://localhost:8000
   - API Docs: http://localhost:8000/docs
   - Frontend: http://localhost:8501

**For evaluation:**
```bash
python evaluation/A2_evaluation.py
```

**For local development:**
```bash
# Run ETL and indexing
python src/A2_etl_chunking.py
python src/A2_indexing.py

# Start API
cd api && uvicorn A2_api:app --reload

# Start frontend
cd frontend && streamlit run A2_frontend.py
```